# Ajustas (Tunear) Hiperparámetros

### ¿Qué son?

* En los modelos de aprendizaje tienen muchos parametros, en los cuales tenemos que establecer la más adecuada, pero esto es muy dificil de saber, entonces podemo recurir a nuestra experiencia o aplicar métodos que nos puedes ayudar con esto.
* Principalmente veremos la de Busqueda en Rejilla y la busqueda de forma aleatoria de hiperparametros.
* Un hiperpárametro podría ser la profundidad de un arbol, pero no sabrías cuál es la más eficiente.
* También se le llama "Tuneo de Modelos".
* Quizas para mejoras el Area bajo la curva.
* ***Modelo de Caja Blanca:*** aquí si podemos ver que proceso o flujo siguió el modelo para ver que modelo tomó, como en los arboles.
* ***Modelo de Caja Negra:*** aquí no se puede ver o deducir, como por ejemplo una regresión lineal de varios tipos.
* En Machine Learning, un hiperparámetro es un valor que no se aprende automáticamente del entrenamiento, sino que lo define el usuario antes de entrenar el modelo.
* El ajuste de hiperparámetros (hyperparameter tuning) es el proceso de buscar los valores óptimos de estos hiperparámetros para que el modelo tenga mejor rendimiento.

### Contexto y Datos

In [1]:
import pandas as pd

personas = pd.read_csv("ingresos.csv")
personas

,edad,estudio,genero,tipo_trabajo,horas,ingreso
0,27,9,0,0,40,0
1,30,9,0,0,40,0
2,34,8,0,0,40,0
3,26,13,0,0,50,0
4,23,4,1,0,25,0
...,...,...,...,...,...,...
95,32,13,1,0,55,1
96,20,10,0,0,25,0
97,46,3,0,0,40,0
98,37,9,1,0,45,0


### Elección de Modelo

* Aquí se crea una instancia del modelo Random Forest de scikit-learn.
* Si no le pasas nada, se inicializa con valores por defecto de hiperparámetros (ejemplo: `n_estimators=100`, `max_depth=None`, etc.)
* `bosque.get_params()`: Este método devuelve un diccionario con todos los hiperparámetros que usa el modelo.
* Cada uno de esos son hiperparámetros configurables que puedes modificar para controlar cómo se entrena el Random Forest.
* ¿Por qué se hace esto?
  - Para ver qué valores por defecto tiene el modelo antes de entrenarlo.
  - Esto es muy útil si después quieres hacer ajuste de hiperparámetros, porque ya sabes qué opciones están disponibles.
* Usar `get_params()` junto con GridSearchCV o RandomizedSearchCV, porque estos métodos necesitan saber qué hiperparámetros puede recibir el modelo.

In [3]:
from sklearn.ensemble import RandomForestClassifier

bosque = RandomForestClassifier()
bosque.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': None,
 'verbose': 0,
 'warm_start': False}

### Búsqueda de Cuadrícula (o Rejilla)

* ¿Qué es la búsqueda en cuadrícula (Grid Search)?
  - Es una técnica sistemática para encontrar la mejor combinación de hiperparámetros de un modelo de Machine Learning.
  - Se define una rejilla (grid) de valores posibles para los hiperparámetros.
  - El algoritmo prueba todas las combinaciones posibles de esos valores.
  - Evalúa el desempeño de cada combinación mediante validación cruzada u otra métrica.
  - Selecciona la combinación con el mejor rendimiento.
* Ventajas
  - Explora de manera sistemática todas las combinaciones.
  - Garantiza que encuentras la mejor configuración posible dentro de la rejilla definida.
* Desventajas
  - Muy costoso computacionalmente, si tienes muchos hiperparametros o los rangos de valores son grandes.

***NOTAS***
* La variable `parametros`, esta tomando que queremos validar entre el criterio de aceptación "gini" o "entropy", estamos pidiendo que verifique entre 10,20 30 árboles, por ultimo que sample los datos, proporción de muestras usadas en cada árbol (1/3 o 2/3 del dataset).
* `rejilla`: este es nuestro objeto tipo GridSearchCV. El `scoring="accuracy`, indica la métrica que se va a optimizar (puedes cambiarla a `precision`, `recall`, `f1`, etc.).
* si no indicas `cv`, por defecto usa `5-fold cross-validation` (validación cruzada).
* En lugar de entrenar directamente el modelo entramos al objeto Grid, pero tiene el detalle que al igual que la validación cruzada solo recibe arreglos de numpy no series de Pandas.
* Salida
  - Atributos del Objeto y el después la mejor configuración.
``` python
GridSearchCV(estimator=RandomForestClassifier(),
             param_grid={'criterion': ('gini', 'entropy'),
                         'max_samples': (0.3333333333333333,
                                         0.6666666666666666),
                         'n_estimators': (10, 20, 30)},
             scoring='accuracy')

### Mejor Modelo.
RandomForestClassifier(criterion='entropy',
                       max_samples=0.3333333333333333,
                       n_estimators=20)
```

In [4]:
from sklearn.model_selection import GridSearchCV

### Deben estar como lo estáran en el modelo
parametros = {
    "criterion" : ("gini","entropy"),
    "n_estimators": (10,20,30),
    "max_samples" : (1/3, 2/3)
}

### Aquí puede ir cualquier metrica
rejilla = GridSearchCV(
    bosque,
    parametros,
    scoring="accuracy"
)

### quitamos la columna variable objetivo
rejilla.fit(
    personas[personas.columns[:-1]].values,
    personas["ingreso"].values
)

GridSearchCV(estimator=RandomForestClassifier(),
             param_grid={'criterion': ('gini', 'entropy'),
                         'max_samples': (0.3333333333333333,
                                         0.6666666666666666),
                         'n_estimators': (10, 20, 30)},
             scoring='accuracy')

### Información de la Búsqueda en Rejilla

* Se hace para explorar qué información guardó GridSearchCV durante el proceso de validación cruzada.
* `rejilla.cv_results_` es un diccionario gigante con todos los resultados de cada combinación de hiperparámetros probada.
* Al hacer `.keys()` obtienes los nombres de las llaves (columnas de información).
* Con `sorted(...)` simplemente ordenas alfabéticamente esas llaves para verlas de manera más limpia.

In [5]:
sorted(rejilla.cv_results_.keys())

['mean_fit_time',
 'mean_score_time',
 'mean_test_score',
 'param_criterion',
 'param_max_samples',
 'param_n_estimators',
 'params',
 'rank_test_score',
 'split0_test_score',
 'split1_test_score',
 'split2_test_score',
 'split3_test_score',
 'split4_test_score',
 'std_fit_time',
 'std_score_time',
 'std_test_score']

### Rejilla de Parámetros Explorados

* `rejilla.cv_results_["params"]`: lo que obtienes es una lista de diccionarios. Cada diccionario corresponde a una combinación de hiperparámetros que GridSearchCV probó.
* Después, esos parámetros se relacionan con los puntajes (mean_test_score, rank_test_score, etc.) para ver cuál fue mejor.

In [6]:
rejilla.cv_results_["params"]

[{'criterion': 'gini', 'max_samples': 0.3333333333333333, 'n_estimators': 10},
 {'criterion': 'gini', 'max_samples': 0.3333333333333333, 'n_estimators': 20},
 {'criterion': 'gini', 'max_samples': 0.3333333333333333, 'n_estimators': 30},
 {'criterion': 'gini', 'max_samples': 0.6666666666666666, 'n_estimators': 10},
 {'criterion': 'gini', 'max_samples': 0.6666666666666666, 'n_estimators': 20},
 {'criterion': 'gini', 'max_samples': 0.6666666666666666, 'n_estimators': 30},
 {'criterion': 'entropy',
  'max_samples': 0.3333333333333333,
  'n_estimators': 10},
 {'criterion': 'entropy',
  'max_samples': 0.3333333333333333,
  'n_estimators': 20},
 {'criterion': 'entropy',
  'max_samples': 0.3333333333333333,
  'n_estimators': 30},
 {'criterion': 'entropy',
  'max_samples': 0.6666666666666666,
  'n_estimators': 10},
 {'criterion': 'entropy',
  'max_samples': 0.6666666666666666,
  'n_estimators': 20},
 {'criterion': 'entropy',
  'max_samples': 0.6666666666666666,
  'n_estimators': 30}]

### Resultados de la Búsqueda en Rejilla

* `rejilla.cv_results_["rank_test_score"]`: Esto devuelve un array con el ranking (1 = mejor, 2 = segundo mejor, etc.) de cada combinación de hiperparámetros que probó.
* `rejilla.cv_results_["mean_test_score"]`: Son los puntajes promedio de validación cruzada (en tu caso, accuracy) para cada combinación de parámetros.

In [7]:
print("="*3, "POSICION DE MEJORIDAD", "="*3)
print(rejilla.cv_results_["rank_test_score"])

print("="*3, "PROMEDIOS CROSS-VALIDACION", "="*3)
print(rejilla.cv_results_["mean_test_score"])

### Puntuación de la mejor Modelo
print(rejilla.best_score_)
### Información de la mejor Modelo
print(rejilla.best_params_)

[ 3  8  5  9  9 11  5  1  4  5 12  2]
[0.82 0.8  0.81 0.78 0.78 0.77 0.81 0.83 0.81 0.81 0.75 0.82]
0.8300000000000001
{'criterion': 'entropy', 'max_samples': 0.3333333333333333, 'n_estimators': 20}


### Usando/Extrayendo el mejor modelo

* `rejilla.predict([[50, 16, 1, 1, 40]])`: Cuando llamas `.predict()`, internamente usa el mejor modelo encontrado `(best_estimator_)` para predecir.
* Entonces este resultado ya es con el mejor bosque.
* `mejor_bosque = rejilla.best_estimator_`: Aquí estás tomando explícitamente el modelo ganador (RandomForestClassifier con los hiperparámetros óptimos) y prediciendo con él.El resultado será exactamente el mismo que el anterior.
* El resultado en este caso será una clasificación.

In [8]:
print(rejilla.predict([[50, 16, 1, 1, 40]]))
mejor_bosque = rejilla.best_estimator_
print(mejor_bosque.predict([[50, 16, 1, 1, 40]]))

[1]
[1]


### Búsqueda Aleatoria

* La búsqueda aleatoria (Random Search) es otra técnica de ajuste de hiperparámetros en Machine Learning, alternativa a la búsqueda en cuadrícula (Grid Search).
* En lugar de probar todas las combinaciones posibles de hiperparámetros como en `Grid Search`, la búsqueda aleatoria elige al azar combinaciones dentro de los rangos definidos.
* Se ejecuta el modelo varias veces con diferentes combinaciones elegidas aleatoriamente, y se guarda la que da mejor resultado.
* Diferencia con Grid Search
  - Grid Search:
    1. Prueba todas las combinaciones de los hiperparámetros.
    2. Muy exhaustivo, pero puede ser muy lento si hay muchos parámetros.
  - Random Search:
    1. Prueba un número fijo de combinaciones al azar.
    2. Más rápido en espacios de búsqueda grandes.
    3. Aunque no garantiza revisar todas las combinaciones, suele encontrar buenos resultados en mucho menos tiempo.

* Ventajas de Random Search
  - Mucho más rápido que Grid Search en espacios grandes.
  - Puede encontrar buenos hiperparámetros sin probar todas las combinaciones.
  - Útil cuando no sabemos qué parámetros son más importantes.
* Desventaja
  - No garantiza encontrar la mejor combinación absoluta, solo una muy buena.

***NOTAS***
* `n_iter=3` → solo se probarán 3 combinaciones al azar, no las 12 posibles.
* 

In [9]:
from sklearn.model_selection import RandomizedSearchCV

parametros = {"criterion": ("gini", "entropy"),
              "n_estimators" : (10, 20, 30), 
              "max_samples" : (1/3, 2/3)}

rejilla_aleatoria = RandomizedSearchCV(bosque, 
                                       parametros,
                                       scoring = "accuracy", 
                                       cv=5, 
                                       n_iter=3)

rejilla_aleatoria.fit(personas[personas.columns[:-1]].values,
                      personas["ingreso"].values)

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(), n_iter=3,
                   param_distributions={'criterion': ('gini', 'entropy'),
                                        'max_samples': (0.3333333333333333,
                                                        0.6666666666666666),
                                        'n_estimators': (10, 20, 30)},
                   scoring='accuracy')

### Parámetros Explorados Aleatoriamente

In [10]:
rejilla_aleatoria.cv_results_["params"]

[{'n_estimators': 10,
  'max_samples': 0.6666666666666666,
  'criterion': 'entropy'},
 {'n_estimators': 10, 'max_samples': 0.6666666666666666, 'criterion': 'gini'},
 {'n_estimators': 20,
  'max_samples': 0.6666666666666666,
  'criterion': 'entropy'}]

### Mejores Parámetros (seleccionados aleatoriamente)

In [11]:
print(rejilla_aleatoria.best_score_)
print(rejilla_aleatoria.best_params_)

0.79
{'n_estimators': 20, 'max_samples': 0.6666666666666666, 'criterion': 'entropy'}
